Cleaning and END for BKK data

In [ ]:
"""
Visualization Module - Budapest Public Transport Wheelchair Accessibility Analysis
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from typing import List, Dict, Set, Tuple
from collections import Counter
import networkx as nx
import sys
import os

# Add project path
sys.path.insert(0, os.path.dirname(__file__))

try:
    from src.gtfs_tools import build_gtfs_network
    from src.api_client import plan_trip
    print("✅ Successfully imported teammate's modules")
except ImportError as e:
    print(f"⚠️ Import failed: {e}")
    print("Attempting to import from Analysis directory...")
    sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))
    from src.gtfs_tools import build_gtfs_network
    from src.api_client import plan_trip

class AccessibilityVisualizer:
    """Wheelchair Accessibility Data Visualizer"""
    
    def __init__(self):
        """Initialize - Load GTFS network data"""
        print("\nLoading GTFS network data...")
        self.network = build_gtfs_network()
        print(f"Loading complete: {len(self.network.routes)} routes, {len(self.network.stops)} stops")
        
    def analyze_stop_accessibility(self) -> Dict:
        """Analyze stop accessibility status"""
        accessible_stops = []
        inaccessible_stops = []
        unknown_stops = []
        
        for stop_id, stop in self.network.stops.items():
            if stop.wheelchair_boarding is True:
                accessible_stops.append(stop)
            elif stop.wheelchair_boarding is False:
                inaccessible_stops.append(stop)
            else:
                unknown_stops.append(stop)
        
        return {
            'accessible': accessible_stops,
            'inaccessible': inaccessible_stops,
            'unknown': unknown_stops,
            'accessible_count': len(accessible_stops),
            'inaccessible_count': len(inaccessible_stops),
            'unknown_count': len(unknown_stops),
            'total': len(self.network.stops)
        }
    
    def plot_accessibility_pie(self):
        """1. Accessibility proportion pie chart"""
        stats = self.analyze_stop_accessibility()
        
        labels = ['Accessible Stops', 'Non-Accessible Stops', 'Unknown Status']
        sizes = [stats['accessible_count'], stats['inaccessible_count'], stats['unknown_count']]
        colors = ['#2ecc71', '#e74c3c', '#95a5a6']
        explode = (0.05, 0, 0)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        wedges, texts, autotexts = ax.pie(
            sizes, explode=explode, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12}
        )
        
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontsize(11)
            autotext.set_fontweight('bold')
        
        ax.set_title('Budapest Public Transport Stop Accessibility Ratio', fontsize=16, fontweight='bold', pad=20)
        
        legend_text = f"Total: {stats['total']:,} stops\nAccessible: {stats['accessible_count']:,}"
        ax.text(1.2, 0.9, legend_text, transform=ax.transAxes, fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        plt.tight_layout()
        plt.savefig('stop_accessibility_pie.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ Saved: stop_accessibility_pie.png")
    
    def plot_route_accessibility_bar(self):
        """2. Accessibility bar chart by transportation mode"""
        route_type_names = {
            0: "Tram",
            1: "Metro",
            3: "Bus"
        }
        
        route_stats = {}
        
        for route_id, route in self.network.routes.items():
            route_type = route.route_type
            route_name = route_type_names.get(route_type, f"Type {route_type}")
            
            if route_name not in route_stats:
                route_stats[route_name] = {
                    'total': 0,
                    'fully_accessible': 0,
                    'partially_accessible': 0,
                    'inaccessible': 0,
                }
            
            route_stats[route_name]['total'] += 1
            
            accessible_count = sum(1 for stop in route.stops if stop.wheelchair_boarding is True)
            total_stops = len(route.stops)
            
            if total_stops > 0:
                accessibility_ratio = accessible_count / total_stops
                if accessibility_ratio == 1.0:
                    route_stats[route_name]['fully_accessible'] += 1
                elif accessibility_ratio > 0:
                    route_stats[route_name]['partially_accessible'] += 1
                else:
                    route_stats[route_name]['inaccessible'] += 1
        
        route_names = list(route_stats.keys())
        fully = [route_stats[r]['fully_accessible'] for r in route_names]
        partially = [route_stats[r]['partially_accessible'] for r in route_names]
        inaccessible = [route_stats[r]['inaccessible'] for r in route_names]
        
        x = np.arange(len(route_names))
        width = 0.25
        
        fig, ax = plt.subplots(figsize=(12, 7))
        
        bars1 = ax.bar(x - width, fully, width, label='Fully Accessible Routes', color='#2ecc71')
        bars2 = ax.bar(x, partially, width, label='Partially Accessible Routes', color='#f39c12')
        bars3 = ax.bar(x + width, inaccessible, width, label='Inaccessible Routes', color='#e74c3c')
        
        ax.set_xlabel('Transportation Mode', fontsize=12)
        ax.set_ylabel('Number of Routes', fontsize=12)
        ax.set_title('Accessibility Distribution by Transportation Mode', fontsize=16, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(route_names)
        ax.legend(loc='upper right', fontsize=10)
        
        for bars in [bars1, bars2, bars3]:
            for bar in bars:
                height = bar.get_height()
                if height > 0:
                    ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('route_accessibility_bar.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ Saved: route_accessibility_bar.png")
    
    def plot_accessibility_map(self):
        """3. Geographic distribution map of accessible stops"""
        stats = self.analyze_stop_accessibility()
        
        fig, ax = plt.subplots(figsize=(14, 10))
        
        accessible_lats, accessible_lons = [], []
        inaccessible_lats, inaccessible_lons = [], []
        unknown_lats, unknown_lons = [], []
        
        for stop in stats['accessible']:
            if stop.lat and stop.lon:
                accessible_lats.append(float(stop.lat))
                accessible_lons.append(float(stop.lon))
        
        for stop in stats['inaccessible']:
            if stop.lat and stop.lon:
                inaccessible_lats.append(float(stop.lat))
                inaccessible_lons.append(float(stop.lon))
        
        for stop in stats['unknown']:
            if stop.lat and stop.lon:
                unknown_lats.append(float(stop.lat))
                unknown_lons.append(float(stop.lon))
        
        ax.scatter(accessible_lons, accessible_lats, c='#2ecc71', s=20, 
                   alpha=0.6, label=f'Accessible Stops ({len(accessible_lats)})', edgecolors='none')
        ax.scatter(inaccessible_lons, inaccessible_lats, c='#e74c3c', s=15,
                   alpha=0.5, label=f'Non-Accessible Stops ({len(inaccessible_lats)})', edgecolors='none')
        ax.scatter(unknown_lons, unknown_lats, c='#95a5a6', s=10,
                   alpha=0.4, label=f'Unknown Status ({len(unknown_lats)})', edgecolors='none')
        
        ax.set_xlabel('Longitude', fontsize=12)
        ax.set_ylabel('Latitude', fontsize=12)
        ax.set_title('Budapest Public Transport Stop Accessibility Distribution Map', fontsize=16, fontweight='bold')
        ax.legend(loc='upper right', fontsize=10, markerscale=2)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.savefig('accessibility_map.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ Saved: accessibility_map.png")
    
    def generate_comprehensive_report(self):
        """Generate comprehensive data report"""
        print("\n" + "="*70)
        print("📊 Budapest Public Transport Accessibility Analysis Report")
        print("="*70)
        
        stop_stats = self.analyze_stop_accessibility()
        print(f"\n📍 Stop Statistics:")
        print(f"   - Accessible stops: {stop_stats['accessible_count']:,} ({stop_stats['accessible_count']/stop_stats['total']*100:.1f}%)")
        print(f"   - Non-accessible stops: {stop_stats['inaccessible_count']:,} ({stop_stats['inaccessible_count']/stop_stats['total']*100:.1f}%)")
        print(f"   - Unknown status: {stop_stats['unknown_count']:,} ({stop_stats['unknown_count']/stop_stats['total']*100:.1f}%)")
        print(f"   - Total: {stop_stats['total']:,}")
        
        print("\n" + "="*70)


def main():
    visualizer = AccessibilityVisualizer()
    visualizer.generate_comprehensive_report()
    
    print("\n📈 Generating visualizations...")
    visualizer.plot_accessibility_pie()
    visualizer.plot_route_accessibility_bar()
    visualizer.plot_accessibility_map()
    
    print("\n🎉 Analysis complete! Charts saved as PNG files.")


if __name__ == "__main__":
    main()